# 1. Class Imbalance & The Accuracy Paradox

### Concept & Definition
Class imbalance occurs when one target class significantly outnumbers another (e.g., 99% Non-Churn vs. 1% Churn, or 99.9% Legitimate vs. 0.1% Fraud).

### The Accuracy Paradox
If a dataset contains 99% negative class samples, a naive dummy model predicting "Negative" for every single observation achieves **99% Accuracy** while completely failing to detect a single positive case (0% Recall).

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("Cleaned_Validated_Data.csv")
df["Target"] = np.where(df["Churn"] == "Yes", 1, 0)

print("=== Target Class Distribution ===")
print(df["Target"].value_counts())
print("\nClass Percentages:")
print(df["Target"].value_counts(normalize=True) * 100)

=== Target Class Distribution ===
Target
0    703
1    307
Name: count, dtype: int64

Class Percentages:
Target
0    69.60396
1    30.39604
Name: proportion, dtype: float64


# 2. Random Undersampling & Oversampling

### Concepts:
- **Random Undersampling:** Deletes random majority class samples to equal minority count. *Risk: Drops valuable data.*
- **Random Oversampling:** Duplicates random minority class samples. *Risk: High overfitting on duplicated rows.*

In [2]:
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler

X = df[["Age", "TenureYears", "MonthlyCharges"]].fillna(0)
y = df["Target"]

# Undersampling
rus = RandomUnderSampler(random_state=42)
X_rus, y_rus = rus.fit_resample(X, y)

# Oversampling
ros = RandomOverSampler(random_state=42)
X_ros, y_ros = ros.fit_resample(X, y)

print(f"Original Shape: {X.shape}, Target Balance: {dict(pd.Series(y).value_counts())}")
print(f"Undersampled Shape: {X_rus.shape}, Target Balance: {dict(pd.Series(y_rus).value_counts())}")
print(f"Oversampled Shape: {X_ros.shape}, Target Balance: {dict(pd.Series(y_ros).value_counts())}")

ModuleNotFoundError: No module named 'imblearn'

# 3. SMOTE & Borderline-SMOTE

### Concepts:
- **SMOTE (Synthetic Minority Over-sampling Technique):** Creates synthetic minority samples along vector lines connecting $k$-nearest neighbors.
- **Borderline-SMOTE:** Focuses synthetic sample generation exclusively near decision boundaries where classification errors occur.

from imblearn.over_sampling import SMOTE, BorderlineSMOTE

# SMOTE
smote = SMOTE(random_state=42)
X_smote, y_smote = smote.fit_resample(X, y)

# Borderline-SMOTE
bsmote = BorderlineSMOTE(random_state=42)
X_bsmote, y_bsmote = bsmote.fit_resample(X, y)

print(f"SMOTE Shape: {X_smote.shape}, Target Counts: {dict(pd.Series(y_smote).value_counts())}")
print(f"Borderline-SMOTE Shape: {X_bsmote.shape}, Target Counts: {dict(pd.Series(y_bsmote).value_counts())}")

# 4. Class Weights (Algorithmic Balancing)

### Concept & Definition
Modifies the loss function during model training by penalizing misclassifications of the minority class proportionally higher without altering sample counts.

In [4]:
from sklearn.utils.class_weight import compute_class_weight

weights = compute_class_weight(class_weight="balanced", classes=np.unique(y), y=y)
class_weight_dict = dict(zip(np.unique(y), weights))

print("Computed Balanced Class Weights:")
print(class_weight_dict)
print("\nNotebook 11 execution completed successfully!")

NameError: name 'y' is not defined